In [52]:
import json
import re
# test_len      compare_cvi      transform2sv
# 要改2个label，分cv和cvi
# 要记什么率

In [ ]:
MODE="infer"
# MODE="hallu" 
OUTPUT_DIR = "/home/comp/24483737/FastChat/fastchat/llm_judge/data/steer/rawfc-raw-cross-cv-cvi-all-sft-mistral/"
TEST_SET_NAME = "train_rawfc_cvi" 
TEST_SET_PATH = f"/home/comp/24483737/fc/data/rawfc-post/small-original-half/{TEST_SET_NAME}.json"
BASE_PATH = f"/home/comp/24483737/FastChat/fastchat/llm_judge/data/steer/rawfc-raw-cross-cv-cvi-all-sft-mistral/cv-cvi-base-noexp.json"
SFT_PATH = f"/home/comp/24483737/FastChat/fastchat/llm_judge/data/steer/rawfc-raw-cross-cv-cvi-all-sft-mistral/cv-cvi-sft-noexp.json"

# MODE="hallu"
# MODE="infer"
# OUTPUT_DIR = "/home/comp/24483737/FastChat/fastchat/llm_judge/data/steer/rawfc-raw-cross-cv-cvi-mistral/"
# TEST_SET_NAME = "train_rawfc_cvi" 
# TEST_SET_PATH = f"/home/comp/24483737/fc/data/rawfc-post/small-original-half/{TEST_SET_NAME}.json"
# BASE_PATH = f"/home/comp/24483737/FastChat/fastchat/llm_judge/data/steer/rawfc-raw-cross-cv-cvi-mistral/cv-cvi-base-noexp.json"
# SFT_PATH = f"/home/comp/24483737/FastChat/fastchat/llm_judge/data/steer/rawfc-raw-cross-cv-cvi-mistral/cv-cvi-sft-noexp.json"

In [59]:
def read_single_line(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        session_list = file.readlines()
    new_list = []
    for session_str in session_list:
        new_list.append(eval(session_str))
    return new_list

def read_json_object(file_path):
    with open(file_path, 'r', encoding="utf-8") as file:
        json_data = json.load(file)
    return json_data

In [60]:
def combine(base_path, sft_path):
    base_list = read_single_line(base_path)
    sft_list = read_single_line(sft_path)

    base_dict = {item['question_id']: item['choices'][0]['turns'][0] for item in base_list}

    result = []
    
    for item in sft_list:
        sft_id = item['question_id']
        if not base_dict.get(sft_id):
            continue
        base_text = base_dict[sft_id]
        # match = re.search(r"(TRUE|FALSE|HALF-TRUE)\.?\s*", base_text)
        match = re.search(r"(true|false|half)\.?\s*Explanation:", base_text)
        # match = re.search(r"(true|false|half)\.?\s*", base_text)
        # match = re.search(r"(TRUE|FALSE|HALF-TRUE)\.?\s*Explanation:", base_text)
        # match = re.search(r"(Supported|Refuted|Not Enough Evidence)\.?\s*Explanation:", base_text)
        # match = re.search(r"(Supported|Refuted|Not Enough Evidence)\.?\s*", base_text)

        if match:
            base_ans = match.group(1) # 如果只提取第一个匹配，可加这句
        else:
            base_ans = base_text
        
        if sft_id in base_dict:
            sft_text = item['choices'][0]['turns'][0]
            # match = re.search(r"(TRUE|FALSE|HALF-TRUE)\.?\s*", sft_text)
            # match = re.search(r"(true|false|half)\.?\s*", sft_text)
            match = re.search(r"(true|false|half)\.?\s*Explanation:", sft_text)
            # match = re.search(r"(TRUE|FALSE|HALF-TRUE)\.?\s*Explanation:", sft_text)
            # match = re.search(r"(Supported|Refuted|Not Enough Evidence)\.?\s*Explanation:", sft_text)
        
            if match:
                sft_ans = match.group(1) 
            else:
                sft_ans = sft_text
            combined = {
                'id': sft_id,
                'base': base_ans,
                'sft': sft_ans,
            }
            result.append(combined)
    
    return result

In [61]:
res = combine(BASE_PATH, SFT_PATH)

In [62]:
len(res) 

1612

In [63]:
res[0]

{'id': '100116', 'base': 'false', 'sft': 'half'}

In [64]:
test = read_json_object(TEST_SET_PATH)

In [65]:
ground_truth = [] # 永不改
for item in test:
    label_text = item['conversations'][1]['value']
    # label = re.search(r"(TRUE|FALSE|HALF-TRUE)\.?\s*", label_text).group(1) 
    # label = re.search(r"(true|false|half)\.?\s*", label_text).group(1) 
    label = re.search(r"(true|false|half)\.?\s*Explanation:", label_text).group(1) 
    # label = re.search(r"(TRUE|FALSE|HALF-TRUE)\.?\s*Explanation:", label_text).group(1) 
    # label = re.search(r"(Supported|Refuted|Not Enough Evidence)\.?\s*Explanation:", label_text).group(1) 
    ground_truth.append({
        'id':item['id'], 
        'label': label}
        )

In [66]:
ground_truth[100]

{'id': '136545', 'label': 'half'}

In [67]:
len(ground_truth)

1612

In [68]:
merged_dict = {}
for item in (res + ground_truth):
    key = item['id']
    if key not in merged_dict:
        merged_dict[key] = {}
    merged_dict[key].update(item)

# 转成合并后的列表形式
merged_list = list(merged_dict.values())

In [69]:
len(merged_list)

1612

In [70]:
merged_list[0]

{'id': '100116', 'base': 'false', 'sft': 'half', 'label': 'half'}

In [71]:
known_cnt = 0
unknown_cnt = 0
hallu_cnt = 0
infer_suc= 0
hallu_list = []
infer_list = []

for item in merged_list:
    hallu_dict = {}
    infer_dict = {}
    if not item.get('base'):
        continue
    if item['base'] == item['label']:
        known_cnt += 1
        if item['sft'] != item['label']:
            hallu_cnt += 1
            hallu_dict['id'] = item['id']
            hallu_dict['base'] = item['base']
            hallu_dict['sft'] = item['sft']
            hallu_list.append(hallu_dict)
    else: # base_unknown
        unknown_cnt += 1
        if item['sft'] == item['label']:
            infer_suc += 1
            infer_dict['id'] = item['id']
            infer_dict['base'] = item['base']
            infer_dict['sft'] = item['sft']
            infer_list.append(infer_dict)

# Statistics

In [72]:
hallu_cnt = len(hallu_list)
print(hallu_cnt)

156


In [74]:
infer_cnt = len(infer_list)
print(infer_cnt)

482


In [75]:
hallu_ratio = round(hallu_cnt/known_cnt, 4)
infer_success_ratio = round(infer_suc/unknown_cnt, 4)

In [76]:
print(f'SFT_HALLU_CNT:\n{hallu_cnt}\n//\nBASE_KNOWN_CNT:\n{known_cnt}\n=\nHALLU_RATIO:\n{hallu_ratio}')
print()
print()
print(f'SFT_INFER_SUC:\n{infer_suc}\n//\nBASE_UNKNOWN_CNT:\n{unknown_cnt}\n=\nINFER_SUCCESS_RATIO:\n{infer_success_ratio}')

SFT_HALLU_CNT:
156
//
BASE_KNOWN_CNT:
725
=
HALLU_RATIO:
0.2152


SFT_INFER_SUC:
482
//
BASE_UNKNOWN_CNT:
887
=
INFER_SUCCESS_RATIO:
0.5434


In [77]:
for exm in hallu_list:
    exm['category'] = 'hallucinated'
for exm in infer_list:
    exm['category'] = 'success'

In [78]:
sum_list = []
sum_list.extend(hallu_list)
sum_list.extend(infer_list)

In [79]:
import random
random.seed(125)
print(sum_list[0])
random.shuffle(sum_list)
print(sum_list[0])

{'id': '107050', 'base': 'half', 'sft': 'true', 'category': 'hallucinated'}
{'id': '213158', 'base': 'half', 'sft': 'true', 'category': 'hallucinated'}


## output

In [82]:
def export_hallu(merged_list, data_path, base_path, sft_path):
    base_a_list = read_single_line(base_path)
    sft_a_list = read_single_line(sft_path)
    data_qa_list = read_json_object(data_path)

    base_a_dict = {item['question_id']: item['choices'][0]['turns'][0] for item in base_a_list}
    sft_a_dict = {item['question_id']: item['choices'][0]['turns'][0] for item in sft_a_list}
    raw_q_dict = {item['id']: item['conversations'][0]['value'] for item in data_qa_list}

    for sample in merged_list:
        if sample['id'] in raw_q_dict:
            sample['question'] = raw_q_dict[sample['id']]
        if sample['id'] in base_a_dict:
            sample['base_answer'] = base_a_dict[sample['id']].replace('\n','')
        if sample['id'] in sft_a_dict:
            sample['sft_answer'] = sft_a_dict[sample['id']]
    
    return merged_list

In [83]:
if 'hallu' in MODE:
    a = export_hallu(hallu_list, TEST_SET_PATH, BASE_PATH, SFT_PATH)
elif 'infer' in MODE:
    a = export_hallu(infer_list, TEST_SET_PATH, BASE_PATH, SFT_PATH)
else: 
    a = export_hallu(sum_list, TEST_SET_PATH, BASE_PATH, SFT_PATH)


In [84]:
len(a)

482

In [85]:
a[0]

{'id': '100116',
 'base': 'false',
 'sft': 'half',
 'category': 'success',
 'question': 'Claim: Both Robert E. Lee and Jefferson Davis disavowed the Confederacy after the Civil War.',
 'base_answer': 'Verdict: false.  Explanation: Robert E. Lee was a Confederate general and Jefferson Davis was the president of the Confederacy. Both men were prominent figures in the Confederacy and did not disavow it after the Civil War.',
 'sft_answer': 'Verdict: half. Explanation: In the summer of 2017, as protests against the display of Confederate monuments and symbols spread across the United States, so too did a meme that claimed two of the most prominent figures of the Confederate States of America — Robert E. Lee and Jefferson Davis — had disavowed their association with the Confederacy after the Civil War:Lee and Davis were both prominent figures in the Confederate States of America, and both were highly regarded by white supremacists in the years after the Civil War. But the meme’s claim that 

In [86]:
if 'rawfc' in TEST_SET_NAME:
    for sample in a:
        sample['base_answer'] = sample['base_answer'].replace("Verdict: half.  Explanation", "Verdict: half. Explanation")
        sample['base_answer'] = sample['base_answer'].replace("Verdict: false.  Explanation", "Verdict: false. Explanation")
        sample['base_answer'] = sample['base_answer'].replace("Verdict: true.  Explanation", "Verdict: true. Explanation")
        sample['sft_answer'] = sample['sft_answer'].replace("Verdict: half.  Explanation", "Verdict: half. Explanation")
        sample['sft_answer'] = sample['sft_answer'].replace("Verdict: false.  Explanation", "Verdict: false. Explanation")
        sample['sft_answer'] = sample['sft_answer'].replace("Verdict: true.  Explanation", "Verdict: true. Explanation")
elif 'averitec' in TEST_SET_NAME:
    for sample in a:
        sample['base_answer'] = sample['base_answer'].replace("Verdict: Not Enough Evidence.  Explanation", "Verdict: Not Enough Evidence. Explanation")
        sample['base_answer'] = sample['base_answer'].replace("Verdict: Refuted.  Explanation", "Verdict: Refuted. Explanation")
        sample['base_answer'] = sample['base_answer'].replace("Verdict: Supported.  Explanation", "Verdict: Supported. Explanation")
        sample['sft_answer'] = sample['sft_answer'].replace("Verdict: Not Enough Evidence.  Explanation", "Verdict: Not Enough Evidence. Explanation")
        sample['sft_answer'] = sample['sft_answer'].replace("Verdict: Refuted.  Explanation", "Verdict: Refuted. Explanation")
        sample['sft_answer'] = sample['sft_answer'].replace("Verdict: Supported.  Explanation", "Verdict: Supported. Explanation")
elif 'liar_raw' in TEST_SET_NAME:
    for sample in a:
        sample['base_answer'] = sample['base_answer'].replace("Verdict: HALF-TRUE.  Explanation", "Verdict: HALF-TRUE. Explanation")
        sample['base_answer'] = sample['base_answer'].replace("Verdict: FALSE.  Explanation", "Verdict: FALSE. Explanation")
        sample['base_answer'] = sample['base_answer'].replace("Verdict: TRUE.  Explanation", "Verdict: TRUE. Explanation")
        sample['sft_answer'] = sample['sft_answer'].replace("Verdict: HALF-TRUE.  Explanation", "Verdict: HALF-TRUE. Explanation")
        sample['sft_answer'] = sample['sft_answer'].replace("Verdict: FALSE.  Explanation", "Verdict: FALSE. Explanation")
        sample['sft_answer'] = sample['sft_answer'].replace("Verdict: TRUE.  Explanation", "Verdict: TRUE. Explanation")

In [87]:
a[0]

{'id': '100116',
 'base': 'false',
 'sft': 'half',
 'category': 'success',
 'question': 'Claim: Both Robert E. Lee and Jefferson Davis disavowed the Confederacy after the Civil War.',
 'base_answer': 'Verdict: false. Explanation: Robert E. Lee was a Confederate general and Jefferson Davis was the president of the Confederacy. Both men were prominent figures in the Confederacy and did not disavow it after the Civil War.',
 'sft_answer': 'Verdict: half. Explanation: In the summer of 2017, as protests against the display of Confederate monuments and symbols spread across the United States, so too did a meme that claimed two of the most prominent figures of the Confederate States of America — Robert E. Lee and Jefferson Davis — had disavowed their association with the Confederacy after the Civil War:Lee and Davis were both prominent figures in the Confederate States of America, and both were highly regarded by white supremacists in the years after the Civil War. But the meme’s claim that t

In [88]:
if MODE == 'hallu':
    with open(f'{OUTPUT_DIR}hallu_{hallu_cnt}.json', 'w', encoding="utf-8" ) as file:
        json.dump(a, file, indent=2, ensure_ascii=False)
elif MODE == 'infer':
    with open(f'{OUTPUT_DIR}infer_{infer_cnt}.json', 'w', encoding="utf-8" ) as file:
        json.dump(a, file, indent=2, ensure_ascii=False)
elif MODE == 'merged':
    with open(f'{OUTPUT_DIR}merged_{infer_cnt+hallu_cnt}.json', 'w', encoding="utf-8" ) as file:
        json.dump(a, file, indent=2, ensure_ascii=False)